# Price Path Research Engine

This notebook demonstrates the research logic behind the Probabilistic Price Path Engine.

It shows how the project:

- loads candle data from a configurable source
- estimates drift and volatility from log returns
- simulates future price paths using GBM and bootstrap methods
- calculates pathwise TP/SL probabilities
- compares Monte Carlo results with an analytical GBM benchmark

The live dashboard uses the same core modules from `src/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

# Resolve project root whether notebook is run from /notebooks or project root
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_loader import load_market_data
from simulator import estimate_mu_sigma, simulate_gbm_paths, simulate_bootstrap_paths
from probability_engine import pathwise_tp_sl_metrics, analytical_gbm_terminal_metrics
from charts import make_price_path_figure

## Configuration

The engine is designed to be data-source agnostic.

The simulation and probability logic only require a standard OHLC dataframe:

```text
time | Open | High | Low | Close | Volume


---

## Config cell

```python
DATA_SOURCE = "synthetic"   # "synthetic", "mt5", or "csv"

SYMBOL = "US100.cash"
TIMEFRAME = "M1"
BARS = 350

CSV_PATH = PROJECT_ROOT / "data" / "historical" / "sample.csv"

TP_POINTS = 29.0
SL_POINTS = 29.0

HORIZON = 20
N_PATHS = 10_000
VOL_WINDOW = 90

DRIFT_MODE = "zero"         # "zero" or "historical"
DIRECTION = "long"          # "long" or "short"

SEED = 42

In [ ]:
if DATA_SOURCE == "csv":
    df = load_market_data(
        source="csv",
        csv_path=CSV_PATH,
    )
else:
    df = load_market_data(
        source=DATA_SOURCE,
        symbol=SYMBOL,
        timeframe=TIMEFRAME,
        bars=BARS,
        synthetic_start_price=21500.0,
        synthetic_seed=SEED,
        synthetic_freq="1min",
    )

df.tail()

## Closed-Candle Research Setup

For live MT5 use, the dashboard is designed to use the latest fully closed candle rather than the currently forming candle.

This keeps the model consistent with candle-close logic and avoids probabilities changing during an unfinished candle.

In [ ]:
USE_CLOSED_CANDLES = DATA_SOURCE == "mt5"

if USE_CLOSED_CANDLES and len(df) > VOL_WINDOW + 21:
    model_df = df.iloc[:-1].copy()
else:
    model_df = df.copy()

model_df.tail()

In [ ]:
current_price = float(model_df["Close"].iloc[-1])

mu, sigma, log_returns = estimate_mu_sigma(
    model_df["Close"],
    window=VOL_WINDOW,
    drift_mode=DRIFT_MODE,
)

summary = {
    "current_price": current_price,
    "mu_per_candle": mu,
    "sigma_per_candle": sigma,
    "vol_window": VOL_WINDOW,
    "drift_mode": DRIFT_MODE,
    "last_model_candle": model_df["time"].iloc[-1],
}

summary

## GBM Monte Carlo Simulation

The GBM path model simulates future prices using:

$$
S_{t+1} = S_t \exp\left((\mu - 0.5\sigma^2) + \sigma Z_t\right)
$$

where:

$$
Z_t \sim \mathcal{N}(0, 1)
$$

In [ ]:
gbm_paths = simulate_gbm_paths(
    current_price=current_price,
    mu=mu,
    sigma=sigma,
    horizon=HORIZON,
    n_paths=N_PATHS,
    seed=SEED,
)

gbm_metrics = pathwise_tp_sl_metrics(
    paths=gbm_paths,
    entry_price=current_price,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

gbm_analytical = analytical_gbm_terminal_metrics(
    current_price=current_price,
    mu=mu,
    sigma=sigma,
    horizon=HORIZON,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

gbm_metrics.update(gbm_analytical)
gbm_metrics

## Historical Bootstrap Simulation

The bootstrap model does not assume normally distributed returns.

Instead, it samples from recent historical log returns and builds paths from those sampled returns:

$$
S_{t+1} = S_t e^{r_t^*}
$$

where $r_t^*$ is sampled from recent empirical returns.

In [ ]:
bootstrap_paths = simulate_bootstrap_paths(
    current_price=current_price,
    historical_returns=log_returns,
    horizon=HORIZON,
    n_paths=N_PATHS,
    sample_window=VOL_WINDOW,
    seed=SEED,
)

bootstrap_metrics = pathwise_tp_sl_metrics(
    paths=bootstrap_paths,
    entry_price=current_price,
    direction=DIRECTION,
    tp_points=TP_POINTS,
    sl_points=SL_POINTS,
)

bootstrap_metrics

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "GBM Monte Carlo",
        "P(TP before SL)": gbm_metrics["p_tp_first"],
        "P(SL before TP)": gbm_metrics["p_sl_first"],
        "P(neither)": gbm_metrics["p_neither"],
        "P(TP touched)": gbm_metrics["p_tp_touched"],
        "P(SL touched)": gbm_metrics["p_sl_touched"],
        "Expected Price": gbm_metrics["expected_price"],
        "Expected Move": gbm_metrics["expected_move"],
        "5%": gbm_metrics["p5"],
        "95%": gbm_metrics["p95"],
    },
    {
        "Model": "Bootstrap",
        "P(TP before SL)": bootstrap_metrics["p_tp_first"],
        "P(SL before TP)": bootstrap_metrics["p_sl_first"],
        "P(neither)": bootstrap_metrics["p_neither"],
        "P(TP touched)": bootstrap_metrics["p_tp_touched"],
        "P(SL touched)": bootstrap_metrics["p_sl_touched"],
        "Expected Price": bootstrap_metrics["expected_price"],
        "Expected Move": bootstrap_metrics["expected_move"],
        "5%": bootstrap_metrics["p5"],
        "95%": bootstrap_metrics["p95"],
    },
])

comparison

In [ ]:
analytical_summary = pd.DataFrame([
    {
        "Metric": "P(up at horizon)",
        "Value": gbm_metrics["gbm_analytical_p_terminal_up"],
    },
    {
        "Metric": "Analytical expected price",
        "Value": gbm_metrics["gbm_analytical_expected_price"],
    },
    {
        "Metric": "Analytical expected move",
        "Value": gbm_metrics["gbm_analytical_expected_move"],
    },
    {
        "Metric": "Analytical terminal TP probability",
        "Value": gbm_metrics["gbm_analytical_terminal_tp_prob"],
    },
    {
        "Metric": "Analytical terminal SL probability",
        "Value": gbm_metrics["gbm_analytical_terminal_sl_prob"],
    },
    {
        "Metric": "Analytical 5% level",
        "Value": gbm_metrics["gbm_analytical_p5"],
    },
    {
        "Metric": "Analytical 95% level",
        "Value": gbm_metrics["gbm_analytical_p95"],
    },
])

analytical_summary

In [ ]:
fig = make_price_path_figure(
    df=model_df,
    paths=gbm_paths,
    metrics=gbm_metrics,
    max_paths=120,
)

fig.show()

In [ ]:
fig = make_price_path_figure(
    df=model_df,
    paths=bootstrap_paths,
    metrics=bootstrap_metrics,
    max_paths=120,
)

fig.show()

## Interpretation Notes

The pathwise Monte Carlo probabilities answer:

```text
Which level is touched first during the simulated path?


---

## Export comparison

```python
output_dir = PROJECT_ROOT / "reports" / "logs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "research_model_comparison.csv"
comparison.to_csv(output_path, index=False)

output_path